# ðŸ“Š Customer Churn Analysis
### Why Customers Leave â€” and What to Do About It

---

**Business Question**
Which customers are most likely to churn, and what can the company do to retain them?

**Dataset**
Telco Customer Churn â€” 7,043 customers, 21 features (IBM sample dataset)

---

## Executive Summary

| | |
|---|---|
| ðŸ”‘ **Key Finding** | Month-to-month customers churn at **42.7%**, vs. just **2.8%** for two-year contracts â€” a ~15x gap, and the strongest churn driver identified in this analysis. |
| ðŸ¤– **Model Performance** | A class-weighted Random Forest model achieves **65.4% recall**, correctly identifying nearly two-thirds of customers who go on to churn. |
| ðŸ’° **Business Impact** | Of ~373 churners in the test set, the model flags **244** for proactive outreach â€” customers who would otherwise receive no warning before cancelling. |
| âœ… **Recommendation** | Prioritize converting month-to-month customers to annual contracts within their first 3 months, and deploy the churn model monthly to flag at-risk customers before they cancel. |

---

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import os

# Works in the repo layout and on Kaggle - adjust the kaggle path to your dataset slug
CANDIDATES = [
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv",  # repo layout
    "data/WA_Fn-UseC_-Telco-Customer-Churn.csv",     # kaggle notebook layout
    "/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv",  # kaggle dataset
]
DATA_PATH = next(p for p in CANDIDATES if os.path.exists(p))
print("loading:", DATA_PATH)
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Missing values after conversion:", df['TotalCharges'].isnull().sum())
print("New dtype:", df['TotalCharges'].dtype)

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print("Missing values now:", df['TotalCharges'].isnull().sum())

In [ ]:
df = df.drop('customerID', axis=1)
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

churn_rate = df['Churn'].value_counts(normalize=True) * 100                              # Overall churn rate
print(churn_rate)

sns.countplot(data=df, x='Churn')
plt.title('Customer Churn Distribution')
plt.show()

In [ ]:
contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack() * 100
print(contract_churn)

plt.figure(figsize=(7,5))
sns.countplot(data=df, x='Contract', hue='Churn')                                  #Churn by contract type
plt.title('Churn by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Number of Customers')
plt.savefig('../outputs/churn_by_contract.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, multiple='stack')
plt.title('Churn Distribution by Tenure')                                              #churn by tenure
plt.xlabel('Tenure (months)')
plt.ylabel('Number of Customers')
plt.savefig('../outputs/churn_by_tenure.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', bins=30, multiple='stack')
plt.title('Churn Distribution by Monthly Charges')                                   #churn by monthly charges
plt.xlabel('Monthly Charges ($)')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(data=df, x='InternetService', hue='Churn')
plt.title('Churn by Internet Service Type')                          #churn by internet service
plt.xlabel('Internet Service')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
df_corr = df.copy()
df_corr['Churn_numeric'] = df_corr['Churn'].map({'Yes': 1, 'No': 0})

plt.figure(figsize=(8,6))                 #Convert Churn to numeric temporarily just for this correlation check     
sns.heatmap(df_corr[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_numeric']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## Key EDA Insights

1. **Contract type is the strongest churn driver.** Month-to-month customers churn at 42.7%, 
   vs 11.3% for one-year and just 2.8% for two-year contracts â€” a ~15x difference.

2. **Early tenure is the highest-risk window.** Churn is heavily concentrated in the first 
   0â€“3 months of a customer relationship, tapering off sharply after that.

3. **Higher monthly charges correlate with more churn.** Customers paying above ~$70/month 
   show a noticeably higher churn proportion than lower-tier customers.

4. **Fiber optic customers churn 2x+ more than DSL customers** (~42% vs ~19%), despite being 
   the premium service â€” suggesting a price-to-value gap worth investigating.

5. **Tenure is the strongest numeric predictor of churn** (correlation: -0.35). Note: tenure 
   and TotalCharges are highly correlated (0.83) â€” only one should be used in modeling to 
   avoid multicollinearity.

In [ ]:
def risk_score(row):
    score = 0
    if row['Contract'] == 'Month-to-month':
        score += 50
    elif row['Contract'] == 'One year':
        score += 20
    else:
        score += 5

    if row['tenure'] <= 3:
        score += 30
    elif row['tenure'] <= 12:
        score += 15                                                 #Create a basic risk score

    if row['MonthlyCharges'] > 70:
        score += 20

    return score

df['RiskScore'] = df.apply(risk_score, axis=1)
df[['tenure', 'Contract', 'MonthlyCharges', 'RiskScore', 'Churn']].head(10)

In [ ]:
def risk_segment(score):
    if score >= 70:
        return 'High Risk'
    elif score >= 40:
        return 'Medium Risk'
    else:
        return 'Low Risk'                                           #create risk segment

df['RiskSegment'] = df['RiskScore'].apply(risk_segment)

# Check how many customers fall into each segment
print(df['RiskSegment'].value_counts())

# Sanity check: does actual churn rate increase as risk segment increases?
print(df.groupby('RiskSegment')['Churn'].value_counts(normalize=True).unstack() * 100)

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='tenure', y='MonthlyCharges', hue='Churn', alpha=0.6)
plt.title('Tenure vs Monthly Charges, Colored by Churn')                 # Visualize the segments
plt.xlabel('Tenure (months)')
plt.ylabel('Monthly Charges ($)')
plt.savefig('../outputs/risk_segment_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_model = df.drop(['RiskScore', 'RiskSegment'], axis=1)

df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})
                                                                   #preparing the data
df_model = pd.get_dummies(df_model, drop_first=True)

df_model.head()

In [ ]:
df_model.shape

In [ ]:
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']
                                                   #spliting the data
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split                      #train/test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                                                      
print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)              #training the model
model.fit(X_train, y_train)

print("Model trained successfully")

In [ ]:
y_pred = model.predict(X_test)
                                           #making predictions
print(y_pred[:10])
print(y_test[:10].values)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))                    #evaluating performance
print("F1 Score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
})

importance = importance.sort_values(by='Coefficient', key=abs, ascending=False)

print(importance.head(10))                 #Feature importance

plt.figure(figsize=(8,6))
sns.barplot(data=importance.head(10), x='Coefficient', y='Feature')
plt.title('Top 10 Features Driving Churn Prediction')
plt.savefig('../outputs/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## Trying a Random Forest Model

Logistic Regression gave us a solid baseline (82% accuracy, 60% recall), but it can only 
capture simple, mostly linear relationships between features and churn. Random Forest builds 
many decision trees and combines their votes, which often captures more complex "if this AND 
that" patterns in customer behavior. Let's train one and compare it directly against our 
Logistic Regression results using the same metrics.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)    #Training Random Forest
rf_model.fit(X_train, y_train)

print("Random Forest trained successfully")

In [ ]:
y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
                                                                     #predict and evaluate
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

## Model Comparison

| Metric | Logistic Regression | Random Forest |
|---|---|---|
| Accuracy | 82.2% | 78.5% |
| Precision | 68.7% | 63.7% |
| Recall | 60.1% | 43.7% |
| F1 Score | 64.1% | 51.8% |

**Logistic Regression outperformed Random Forest on every metric**, particularly recall â€” 
likely due to Random Forest's default settings being more conservative on the minority 
(churn) class given the dataset's ~73/27 class imbalance. **Logistic Regression is selected 
as the final model** for this project.

## Improving Random Forest with Class Weighting

Random Forest underperformed on recall, likely because it treated both classes equally 
despite the ~73/27 imbalance. Let's try `class_weight='balanced'`, which tells the model to 
pay proportionally more attention to the minority class (churned customers) during training, 
and see if that recovers the lost recall.

In [ ]:
rf_balanced = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_balanced.fit(X_train, y_train)

y_pred_rf_balanced = rf_balanced.predict(X_test)
                                                                    #Retrain with class_weight='balanced'
print("Accuracy:", accuracy_score(y_test, y_pred_rf_balanced))
print("Precision:", precision_score(y_test, y_pred_rf_balanced)) 
print("Recall:", recall_score(y_test, y_pred_rf_balanced))
print("F1 Score:", f1_score(y_test, y_pred_rf_balanced))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_balanced))

## Model Comparison (Updated)

| Metric | Logistic Regression | Random Forest (default) | Random Forest (balanced) |
|---|---|---|---|
| Accuracy | 82.2% | 78.5% | 78.9% |
| Precision | 68.7% | 63.7% | 59.2% |
| Recall | 60.1% | 43.7% | 65.4% |
| F1 Score | 64.1% | 51.8% | 62.2% |

**Random Forest with `class_weight='balanced'` achieves the best recall (65.4%)**, catching 
the most actual churners at the cost of some precision (more false alarms). Since missing an 
at-risk customer is more costly to the business than a wasted retention offer, **the balanced 
Random Forest is selected as the final model** for this project.

## Business Recommendations

**1. Incentivize month-to-month customers to switch to annual contracts.**
Month-to-month customers churn at 42.7% vs just 2.8% for two-year contracts â€” a ~15x gap, 
and the single strongest predictor in both our EDA and model. Offering a modest discount 
(e.g., 10-15%) to convert a customer to a one-year contract within their first 3 months 
would likely pay for itself many times over in retained revenue.

**2. Focus retention efforts on the first 3 months of the customer lifecycle.**
Churn is heavily concentrated in early tenure. A structured onboarding check-in call or 
email at day 30 and day 60 could catch dissatisfaction before it becomes a cancellation.

**3. Investigate the Fiber optic service experience.**
Fiber optic customers churn 2x+ more than DSL customers despite paying a premium â€” a gap 
between price and perceived value. This warrants a closer look at fiber-specific complaints, 
pricing tiers, or competitor offers targeting this segment specifically.

**4. Bundle Online Security and Tech Support into base plans for high-risk segments.**
Customers with these add-ons churn noticeably less, suggesting these services increase 
perceived value or "stickiness." Offering them free for the first 3 months to new 
month-to-month customers could reduce early churn.

**5. Deploy the churn model to proactively flag at-risk customers monthly.**
Using the balanced Random Forest model (65.4% recall), the retention team could receive a 
monthly list of flagged high-risk customers to prioritize outreach â€” rather than waiting for 
a cancellation request.

## Exporting Cleaned Data for Power BI

To build an interactive dashboard alongside this notebook, we'll export the cleaned dataset 
(with fixed data types and risk segments) as a CSV that Power BI can import directly.

In [ ]:
df.to_csv('telco_churn_with_segments.csv', index=False)
print("Exported successfully")